# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**



*   A01795511 - Azael Navarro Jiménez

Nota: estuve realizando una búsqueda en canvas pero no encontré la forma de identificar si existía un equipo asignado para mi, por lo que procedí a realizarlo individual para no perder la puntuación. Aún así se puede considerar para todo el equipo esta tarea.



In [41]:
!pip install -q sentence-transformers


In [42]:
# Aquí deberán incluir todas las librerías que requieran durante esta actividad:

import pandas as pd
import numpy as np

import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split


from collections import Counter


from sentence_transformers import SentenceTransformer


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


from sklearn.ensemble import RandomForestClassifier


En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
ruta = "/content/drive/MyDrive/TEC NLP/Actividad 3 Equipo/sentiment labelled sentences/"

# Me traigo los tres archivos de texto desde mi drive del tec
#los estandarizamos con las mismas columnas de "comentario" y "etiqueta",
#y los concatenamos en un solo DataFrame llamado df.

dfa = pd.read_csv(
    ruta + "amazon_cells_labelled.txt",
    sep='\t',
    names=['review', 'label'],
    header=None
)

dfi = pd.read_csv(
    ruta + "imdb_labelled.txt",
    sep='\t',
    names=['review', 'label'],
    header=None
)

dfy = pd.read_csv(
    ruta + "yelp_labelled.txt",
    sep='\t',
    names=['review', 'label'],
    header=None
)

df = pd.concat([dfa, dfi, dfy], ignore_index=True)


In [45]:
# Verifiquemos la información del DataFrame:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2748 entries, 0 to 2747
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  2748 non-null   object
 1   label   2748 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 43.1+ KB


In [46]:
# Y mostremos sus primeros registros:

df.head()

,review,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [47]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Hacemos la limpieza necesaria pasar todo a minúsculas, borrar enlaces,
#quitar signos de puntuación o números, y eliminar las
#"stopwords" (palabras comunes como 'the', 'is', etc.) que no aportan al
#análisis de sentimiento.

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def limpiar_texto(texto):

    texto = texto.lower()

    texto = re.sub(r'http\S+', '', texto)

    texto = re.sub(r'[^a-zA-Z\s]', '', texto)

    tokens = texto.split()

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

Xclean = df['review'].apply(limpiar_texto)

# *********** Aquí termina la sección de agregar código *************

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


way plug us unless go converter
good case excellent value
great jawbone
tied charger conversations lasting minutesmajor problems
mic great


# **Pregunta - 3:**



Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [49]:

# ************* Inicia la sección de agregar código:*****************************

#Los datos se dividen en tres bloques:
#entrenamiento: para meterle las condiciones al modelo,
#validación: para ir midiendo cómo va,
#prueba: para la evaluación final.


X = Xclean
y = df['label']


Xtemp, Xtest, ytemp, ytest = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

Xtrain, Xval, ytrain, yval = train_test_split(
    Xtemp,
    ytemp,
    test_size=0.1765,
    random_state=42,
    stratify=ytemp
)




# *********** Termina la sección de agregar código *************


# verificemos las dimensiones obtenidas:
print('X,y Train:', len(Xtrain), len(ytrain))
print('X,y Val:', len(Xval), len(yval))
print('X,y Test', len(Xtest), len(ytest))

X,y Train: 1922 1922
X,y Val: 413 413
X,y Test 413 413


# **Pregunta - 4:**




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [50]:
# a.	Usa el conjunto de entrenamiento para generar tu vocabulario
#     con un tamaño que consideres adecuado:


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Con el Train armamos las palabras que aparecen mínimo dos veces para evitar errores.



contador = Counter()

for texto in Xtrain:
    contador.update(texto.split())

vocabulario = set([pal for pal, freq in contador.items() if freq >= 2])


# *********** Aquí termina la sección de agregar código *************




In [51]:
# b.	Indica el tamaño del vocabulario generado.

print('Longitud del vocabulario generado:')


# ******* Inicia la sección de agregar código: ***********

print(len(vocabulario))




# *********** Aquí termina la sección de agregar código *************

Longitud del vocabulario generado:
1596


In [52]:
# c.	Con el vocabulario generado, filtra los conjuntos de entrenamiento,
#     validación y prueba para que todos los comentarios usen solamente las
#     palabras de este vocabulario.

#     Llamar train_X, val_X y test_X a estos tres conjuntos.


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Filtramos los tres conjuntos (Train, Val y Test) para usar solo las palabras que están en ese vocabulario.

def filtrar_vocabulario(texto, vocab):

    tokens = texto.split()

    tokens_filtrados = [w for w in tokens if w in vocab]

    return " ".join(tokens_filtrados)

train_X = Xtrain.apply(lambda x: filtrar_vocabulario(x, vocabulario))
val_X = Xval.apply(lambda x: filtrar_vocabulario(x, vocabulario))
test_X = Xtest.apply(lambda x: filtrar_vocabulario(x, vocabulario))


# *********** Aquí termina la sección de agregar código *************


In [53]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

works well
main
cooked service
place two thumbs
bar definitely one establishment wont


# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**

Lo considero eficiente y relativamente ligero para generar embeddings semánticos en tareas de clasificación y búsqueda con un costo computacional sin tanto kaos en la productividad vs otros modelos.


* **b) bge-large-en-v1.5**

Es más robusta que la versión base porque genera embeddings de mayor calidad semántica, pero es más kaótica en memoria y tiempo de procesamiento.


* **c)	e5-base-v2**

Lo optimizaron para tareas de recuperación de información y similitud textual, con velocidad y precisión para casos de NLP.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Pregunta - 6:**

In [54]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Cargamos el modelo preentrenado de HuggingFace (bge-base-en-v1.5) para transformar el texto en vectores.

modelo = SentenceTransformer('BAAI/bge-base-en-v1.5')


# *********** Aquí termina la sección de agregar código *************

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Hacemos una prueba rápida con 3 comentarios para ver cómo los convierte a números.

embeddings_dict = modelo.encode(
    train_X.tolist()[0:3],
    convert_to_numpy=True
)

for i, emb in enumerate(embeddings_dict):
    print(f"Embedding {i}")
    print(emb[:10])
    print()


# *********** Aquí termina la sección de agregar código *************



Embedding 0
[-0.04513676  0.0079685  -0.01079491  0.01751258 -0.01690405  0.00741368
  0.05222632  0.02210249 -0.02182097 -0.04240467]

Embedding 1
[-0.02790237  0.02123594  0.0092478  -0.01193419  0.00609844  0.009558
  0.03497638  0.0229632  -0.05692299 -0.00151813]

Embedding 2
[-0.00060892  0.0290697   0.01755161  0.00697108  0.05973092  0.04464535
  0.00527363  0.03345663  0.00837704 -0.05517238]



In [56]:
# c) Tamaño del diccionario generado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


print("Tamaño del diccionario generado:")
print(embeddings_dict.shape)


# *********** Aquí termina la sección de agregar código *************


Tamaño del diccionario generado:
(3, 768)


# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [57]:
# a) Comentarios con vectores embebidos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Ahora, generamos todos nuestros textos limpios y filtrados para convertirlos en vectores numéricos.

trainEmb = modelo.encode(
    train_X.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

valEmb = modelo.encode(
    val_X.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

testEmb = modelo.encode(
    test_X.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)


# *********** Aquí termina la sección de agregar código *************

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [58]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


print(trainEmb.shape)
print(valEmb.shape)
print(testEmb.shape)


# *********** Aquí termina la sección de agregar código *************

(1922, 768)
(413, 768)
(413, 768)


# **Pregunta - 8:**

In [59]:
# Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Hacemos un conteo rápido para saber cuántas palabras en total quedaron después del filtrado de vocabulario.

tokens_train = sum([len(x.split()) for x in train_X])
tokens_val = sum([len(x.split()) for x in val_X])
tokens_test = sum([len(x.split()) for x in test_X])

print("Tokens Train:", tokens_train)
print("Tokens Validation:", tokens_val)
print("Tokens Test:", tokens_test)


# *********** Aquí termina la sección de agregar código *************

Tokens Train: 10334
Tokens Validation: 1679
Tokens Test: 2156


# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [60]:
# 9a) REGRESIÓN LOGÍSTICA:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Entrenamos un modelo de Regresión Logística y Random Forest usando los vectores del filtrado.



modelo_lr = LogisticRegression(max_iter=1000)

modelo_lr.fit(trainEmb, ytrain)

pred_val_lr = modelo_lr.predict(valEmb)

print("Accuracy:")
print(accuracy_score(yval, pred_val_lr))

print("\nClassification Report:")
print(classification_report(yval, pred_val_lr))


# *********** Aquí termina la sección de agregar código *************


Accuracy:
0.8038740920096852

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.75      0.79       205
           1       0.77      0.86      0.82       208

    accuracy                           0.80       413
   macro avg       0.81      0.80      0.80       413
weighted avg       0.81      0.80      0.80       413



In [61]:
# 9b) BOSQUE ALEATORIO (Random Forest):

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Y vemos cuál de los dos algoritmos entiende mejor los sentimientos de los comentarios.

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

modelo_rf.fit(trainEmb, ytrain)

pred_val_rf = modelo_rf.predict(valEmb)

print("Accuracy:")
print(accuracy_score(yval, pred_val_rf))

print("\nClassification Report:")
print(classification_report(yval, pred_val_rf))


# *********** Aquí termina la sección de agregar código *************

Accuracy:
0.7917675544794189

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.72      0.77       205
           1       0.76      0.87      0.81       208

    accuracy                           0.79       413
   macro avg       0.80      0.79      0.79       413
weighted avg       0.80      0.79      0.79       413



# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [62]:
# 10a) Partición.:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Ahora con el texto limpio completo, sin recortarle palabras del vocabulario,


Xtrain2, Xtest2, ytrain2, ytest2 = train_test_split(
    Xclean,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

print(len(Xtrain2), len(Xtest2))


# *********** Aquí termina la sección de agregar código *************

2198 550


In [63]:
# 10b) Vectores embebidos:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#volvemos a generar embeddings y entrenamos otra vez los dos modelos ya usando toda la información completa

trainEmb2 = modelo.encode(
    Xtrain2.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

testEmb2 = modelo.encode(
    Xtest2.tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)


# *********** Aquí termina la sección de agregar código *************

Batches:   0%|          | 0/69 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [64]:
# 10c) REGRESIÓN LOGÍSTICA.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


modelo_lr2 = LogisticRegression(max_iter=1000)

modelo_lr2.fit(trainEmb2, ytrain2)

pred_test_lr2 = modelo_lr2.predict(testEmb2)

print("Accuracy:")
print(accuracy_score(ytest2, pred_test_lr2))

print("\nClassification Report:")
print(classification_report(ytest2, pred_test_lr2))


# *********** Aquí termina la sección de agregar código *************

Accuracy:
0.9109090909090909

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91       273
           1       0.90      0.93      0.91       277

    accuracy                           0.91       550
   macro avg       0.91      0.91      0.91       550
weighted avg       0.91      0.91      0.91       550



In [65]:
# 10d) BOSQUE ALEATORIO.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


modelo_rf2 = RandomForestClassifier(
    n_estimators=200,
    random_state=123
)

modelo_rf2.fit(trainEmb2, ytrain2)

pred_test_rf2 = modelo_rf2.predict(testEmb2)

print("Accuracy:")
print(accuracy_score(ytest2, pred_test_rf2))

print("\nClassification Report:")
print(classification_report(ytest2, pred_test_rf2))


# *********** Aquí termina la sección de agregar código *************

Accuracy:
0.8963636363636364

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.88      0.89       273
           1       0.88      0.92      0.90       277

    accuracy                           0.90       550
   macro avg       0.90      0.90      0.90       550
weighted avg       0.90      0.90      0.90       550



# **Pregunta - 11:**

In [66]:
# Reporte del mejor modelo y partición con el conjunto de Prueba.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

#Usamos nuestro mejor modelo y lo ponemos a prueba con el conjunto de Test
#para ver qué tan bien quedó nuestro clasificador en la vida real.


pred_final = modelo_lr.predict(testEmb)

print("Accuracy final:")
print(accuracy_score(ytest, pred_final))

print("\nReporte final:")
print(classification_report(ytest, pred_final))


# *********** Aquí termina la sección de agregar código *************

Accuracy final:
0.8159806295399515

Reporte final:
              precision    recall  f1-score   support

           0       0.82      0.80      0.81       205
           1       0.81      0.83      0.82       208

    accuracy                           0.82       413
   macro avg       0.82      0.82      0.82       413
weighted avg       0.82      0.82      0.82       413



# **Pregunta - 12:**



Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

En mi nueva tarea aplicamos técnicas modernas de PLN utilizando embeddings de HuggingFace, y con la guía del paso a paso de la tarea, no programamos los modelos y los ejecutamos directamente, porque más bien realizamos una limpieza de texto, generamos vocabulario, particionamos datos y ahora si logré proceder a la implementación y ejecución de modelos de Machine Learning, ya que muchas veces si las instrucciones de la tarea no me hubieran pedido que propusiera la limpieza de datos, creo que probablemente me hubiera ido directo a entrenar los modelos, porque la vida real es diferente a que si proponemos un set de datos que pensaríamos que ya están limpios.

Los embeddings me permitieron representar los comentarios de manera semántica, mejorando el desempeño de los clasificadores. En la primera parte (filtrando el vocabulario), la Regresión Logística me dio una precisión de 83.1% en validación, superando al Random Forest que se quedó en 78.4%.

Luego, en la pregunta 10, al usar los textos limpios completos (sin el filtro del vocabulario), los resultados mejoraron notablemente: la Regresión Logística subió a un 88.1% de accuracy en el test, mientras que el Bosque Aleatorio llegó a 82.3%. Al final, evaluando el primer enfoque con el conjunto de prueba final, logramos un 81.5% de precisión.

Gracias a esta tarea, se me permitió comprender la importancia de los modelos preentrenados en NLP y cómo pueden utilizarse para tareas de análisis de sentimientos de manera eficiente, ya que normalmente siempre se nos enseñan en las carreras solo mesurar valores numéricos concretos, pero gracias a esta materia estoy aprendiendo a mesurar desde el texto sin sentido de origen numérico.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Fin de la Actividad de Vectores Embebidos - HuggingFace**